# DataCleaning & Exploration

This notebook follows a **data science workflow** on the combined food dataset:

1. **Data loading** — Read the combined index and labels.
2. **Data cleaning** — Missing paths, duplicates, corrupt images, label consistency.
3. **Exploratory Data Analysis (EDA)** — Class balance, splits, sample images, basic stats.

Output: a clean, understood dataset ready for MobileNetV2 (or any CNN) training.

---
## 1. Setup & load data

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

BASE = Path(".").resolve()
DATA_DIR = BASE / "combined_dataset"
INDEX_CSV = DATA_DIR / "combined_index.csv"
LABELS_JSON = DATA_DIR / "labels.json"
LABEL2ID_JSON = DATA_DIR / "label2id.json"

assert DATA_DIR.exists(), f"Run combine_datasets.ipynb first. Missing: {DATA_DIR}"
assert INDEX_CSV.exists(), f"Missing: {INDEX_CSV}"

df = pd.read_csv(INDEX_CSV)
with open(LABELS_JSON, "r", encoding="utf-8") as f:
    id2label = json.load(f)
with open(LABEL2ID_JSON, "r", encoding="utf-8") as f:
    label2id = json.load(f)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(10)

Shape: (54175, 6)
Columns: ['path', 'label', 'label_id', 'source', 'split', 'path_type']


,path,label,label_id,source,split,path_type
0,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
1,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
2,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
3,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
4,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
5,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
6,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
7,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
8,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file
9,C:\Users\MSI GF63\Desktop\DatasetRasaRight\com...,Fish And Chips,22,Malaysia_Food_11,train,file


---
## 2. Data cleaning

### 2.1 Missing values & path checks

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()

# Path exists?
df["path_exists"] = df["path"].apply(lambda p: Path(p).exists())
missing_paths = (~df["path_exists"]).sum()
print(f"Paths that do not exist on disk: {missing_paths}")

if missing_paths > 0:
    print("Sample missing paths:")
    print(df.loc[~df["path_exists"], "path"].head())
    df = df[df["path_exists"]].copy()
    df = df.drop(columns=["path_exists"])
else:
    df = df.drop(columns=["path_exists"])
print(f"Rows after dropping missing paths: {len(df)}")

### 2.2 Duplicate paths

In [ ]:
dup_paths = df["path"].duplicated(keep="first")
n_dup = dup_paths.sum()
print(f"Duplicate paths: {n_dup}")

if n_dup > 0:
    df = df[~dup_paths].copy()
    print(f"Rows after dropping duplicates: {len(df)}")
else:
    print("No duplicates.")

### 2.3 Label consistency (label_id vs label2id)

In [ ]:
id2label_int = {int(k): v for k, v in id2label.items()}
df["label_expected"] = df["label_id"].map(id2label_int)
label_ok = (df["label"] == df["label_expected"])
print(f"Rows where label matches id2label: {label_ok.sum()} / {len(df)}")

if (~label_ok).any():
    print("Mismatches:")
    print(df.loc[~label_ok, ["path", "label", "label_id", "label_expected"]].head())
df = df.drop(columns=["label_expected"])

### 2.4 Corrupt / unreadable images

In [ ]:
def image_readable(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

print("Checking a sample of paths for readability (this may take a minute)...")
sample_size = min(2000, len(df))
sample_idx = np.random.default_rng(42).choice(len(df), size=sample_size, replace=False)
readable = []
for i in tqdm(sample_idx, desc="Verify"):
    readable.append(image_readable(df.iloc[i]["path"]))

n_bad = sum(1 for r in readable if not r)
print(f"In sample of {sample_size}: {n_bad} unreadable images.")

if n_bad > 0:
    bad_idx = [sample_idx[i] for i in range(sample_size) if not readable[i]]
    print("Example bad paths:", df.iloc[bad_idx[:3]]["path"].tolist())
    print("To remove all corrupt: run full scan and filter (optional, slow).")
else:
    print("Sample passed; assume rest are OK or run full scan for certainty.")

### 2.5 Class balance (minimum samples per class)

In [ ]:
counts = df["label"].value_counts()
min_per_class = 5  # optional: drop classes with very few samples
rare = counts[counts < min_per_class]
print(f"Classes with < {min_per_class} samples: {len(rare)}")
if len(rare) > 0:
    print(rare)
    # Optional: uncomment to drop rare classes
    # df = df[df["label"].isin(counts[counts >= min_per_class].index)].copy()
    # print(f"Rows after dropping rare classes: {len(df)}")

### 2.6 Train / Val / Test split (80% / 10% / 10%)

Stratified split so each set has the same class proportions. Updates the `split` column and saves `combined_index.csv`.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified 80% train, 20% temporary (then split into 10% val, 10% test)
train_idx, temp_idx = train_test_split(
    df.index, test_size=0.2, stratify=df["label"], random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=df.loc[temp_idx, "label"], random_state=42
)

df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

print("Split counts:", df["split"].value_counts().to_dict())
print("Fractions: train {:.1%}, val {:.1%}, test {:.1%}".format(
    (df["split"] == "train").mean(),
    (df["split"] == "val").mean(),
    (df["split"] == "test").mean(),
))

# Persist split to combined_dataset so downstream use 80/10/10
out_cols = [c for c in df.columns if c in ["path", "label", "label_id", "source", "split", "path_type"]]
df_out = df[out_cols] if out_cols else df
df_out.to_csv(INDEX_CSV, index=False)
print("Updated:", INDEX_CSV)

### 2.7 Clean dataset summary

In [ ]:
print("Clean dataset:")
print(f"  Total images: {len(df)}")
print(f"  Classes: {df['label'].nunique()}")
print(f"  Splits: {df['split'].value_counts().to_dict()}")
df_clean = df.copy()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Class distribution

In [ ]:
counts = df_clean["label"].value_counts()
fig, ax = plt.subplots(figsize=(14, 6))
counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="navy", alpha=0.8)
ax.set_title("Number of images per class (food)")
ax.set_xlabel("Class (label)")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Top 10 classes:")
print(counts.head(10))
print("\nBottom 5 classes:")
print(counts.tail(5))

### 3.2 Split distribution (train / validation / test)

In [ ]:
split_counts = df_clean["split"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(split_counts, labels=split_counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("Split distribution")
plt.tight_layout()
plt.show()
print(split_counts)

### 3.3 Source distribution (which dataset each image came from)

In [ ]:
src_counts = df_clean["source"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
src_counts.plot(kind="bar", ax=ax, color=["coral", "seagreen", "gold"])
ax.set_title("Images per source dataset")
ax.set_xlabel("Source")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(src_counts)

### 3.4 Source × Class (heatmap sample)

In [ ]:
cross = pd.crosstab(df_clean["label"], df_clean["source"])
# Show top 15 classes by total count
top_classes = df_clean["label"].value_counts().head(15).index
cross_top = cross.loc[cross.index.isin(top_classes)]
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(cross_top, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("Top 15 classes: count by source")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.5 Sample images per class

In [ ]:
n_classes_show = 8
n_per_class = 3
classes_show = df_clean["label"].value_counts().head(n_classes_show).index.tolist()

fig, axes = plt.subplots(n_classes_show, n_per_class, figsize=(n_per_class * 3, n_classes_show * 3))
for i, cls in enumerate(classes_show):
    sub = df_clean[df_clean["label"] == cls].sample(n=min(n_per_class, len(df_clean[df_clean["label"] == cls])), random_state=42)
    for j, (_, row) in enumerate(sub.iterrows()):
        ax = axes[i, j]
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "Load error", ha="center", va="center")
        ax.set_title(cls if j == 0 else "")
        ax.axis("off")
plt.suptitle("Sample images per class (top 8 classes)", y=1.02)
plt.tight_layout()
plt.show()

### 3.6 Image stats (size, mean pixel)

In [ ]:
print("Computing basic stats on a sample of images...")
sample_n = min(500, len(df_clean))
sample_df = df_clean.sample(n=sample_n, random_state=42)
sizes = []
means = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Stats"):
    try:
        img = np.array(Image.open(row["path"]).convert("RGB"))
        sizes.append(img.shape)
        means.append(img.mean())
    except Exception:
        pass

if sizes:
    unique_shapes = set(sizes)
    print(f"Unique shapes in sample: {unique_shapes}")
    print(f"Mean pixel value (sample): {np.mean(means):.1f}")
    print("All images should be 224x224 from the combine notebook.")
else:
    print("No valid images in sample.")

### 3.7 Summary for modeling

In [ ]:
print("=== EDA Summary ===")
print(f"Total samples: {len(df_clean)}")
print(f"Number of classes: {df_clean['label'].nunique()}")
print(f"Label IDs: 0 to {df_clean['label_id'].max()}")
print(f"Image size: 224 x 224 (from combine step)")
counts = df_clean["label"].value_counts()
print(f"\nClass imbalance: max={counts.max()}, min={counts.min()}, ratio={counts.max()/max(counts.min(),1):.1f}x")
print("\nNext: use df_clean and label2id/id2label for MobileNetV2 training (train/val split by 'split' or sklearn).")